# S11 · leakdepth 0.1.1 — further validation and reporting (NB5)

Runs the protocol pre-registered as amendment **A12** before this notebook was built. leakdepth 0.1.1 (same wheel as NB3)
uses the post-hoc minimum-information lineage rule A10 by default (20 bits); `matcher="frozen"` is the 0.1.0 matcher.
Nothing is re-tuned on these results.

| Cell | Item | What it does |
|---|---|---|
| 2 | E | gate: 0.1.1 reproduces the NB3 counts on the Ahuja–SNT control (frozen and A10) |
| 3 | N | new negative controls not used to design A10 (Hugging Face corpora pinned by revision; optional SDN-DDoS Traffic from Drive), with an exact-overlap gate |
| 4 | T | threshold-free lineage measure on seeds 100–119 and 200–219 (containment with interval vs true planted share, row recall/precision, detection across thresholds) |
| 5 | B | group-bootstrap 95 % intervals for the sensitivity results, read from the saved NB1 reports (no retraining) |
| 6 | X | the cross-tool harness's leakdepth runner rerun with 0.1.1 on seeds 100–109 |
| 7 | Summary | compact JSON (paste it back) |

**Runtime.** Colab **High-RAM CPU**; roughly 1–2 h (downloads ≈ 1.6 GB). Outputs go to `MyDrive/S11-SecSplit/nb5/` as each job
finishes. After a disconnect rerun cells 1–2, then the interrupted cell; finished jobs are skipped.


In [ ]:
# 1. Setup: Drive, embedded leakdepth 0.1.1 wheel (hash-checked), embedded harness files, versions
import base64, hashlib, json, os, subprocess, sys, time, glob, gc, zipfile, io, math, importlib, re
LOCAL = os.environ.get("NB5_LOCAL")                     # local smoke tests only; unset on Colab
SMOKE = bool(os.environ.get("NB5_SMOKE"))
if LOCAL:
    BASE = LOCAL
else:
    if not os.path.isdir("/content/drive/MyDrive"):     # mount once; re-running cells never re-prompts
        from google.colab import drive
        drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
OUT = f"{BASE}/S11-SecSplit/nb5"
NB1 = f"{BASE}/S11-SecSplit/nb1"
RAW6 = f"{BASE}/S06-OFDG/raw"
os.makedirs(OUT, exist_ok=True)
WD = LOCAL or "/content"
WHEEL_SHA = "13c0954a6ba000c65a44cad280a8f1e59f1fdab63fd7c1048e9a95a9dc617eb6"  # package files identical to the tested wheel 5213ebdf23c2…; author metadata set to the release author
WHEEL_B64 = "..."
BENCH = {'compare.py': {'sha256': 'eb4ce5c31818339e4ad99afa90d2b17225a179661f3b1a8769688c58043fede4', 'b64': 'IiIiQ3Jvc3MtdG9vbCBjb21wYXJpc29uIG9uIHRoZSBpbmplY3Rpb24gYmVuY2htYXJrLgoKRWFjaCB0b29sIHJ1bnMgaW4gaXRzIG93biB2aXJ0dWFsIGVudmlyb25tZW50IGFzIGEgbG9uZy1saXZlZCB3b3JrZXIgcHJvY2VzcwooYmVuY2gvY29tcGFyZS9ydW5fPHRvb2w+LnB5KSB0aGF0IHJlYWRzIGpvYnMgYXMgSlNPTiBsaW5lcy4gVG9vbHMgYXJlIGdpdmVuIHRoZQpzYW1lIENTViBmaWxlcy4gQ29uZGl0aW9uczoKICBkZWZhdWx0cyAtIHRyYWluL3Rlc3QgZnJhbWVzIGFzIGdlbmVyYXRlZCAoZmVhdHVyZXMgKyBtZXRhZGF0YSArIGxhYmVsKTsKICBmZWF0dXJlcyAtIGZlYXR1cmUgY29sdW1ucyArIGxhYmVsIG9ubHkgKG1ldGFkYXRhIGRyb3BwZWQpOwogIG9yYWNsZSAgIC0gZGVmYXVsdHMgcGx1cyB0aGUgdHJ1ZSB1bml0IGNvbHVtbiBgX3VuaXRgLCBwYXNzZWQgYXMgdGhlIGdyb3VwCiAgICAgICAgICAgICBjb2x1bW4gdG8gdG9vbHMgdGhhdCBhY2NlcHQgb25lLgoKdXNhZ2U6IHB5dGhvbiBiZW5jaC9jb21wYXJlLnB5IC0tdG9vbHMtZGlyIC4uL3Rvb2xzIC0tc2VlZHMgMTAwLTEwOSAtLW91dCByZXN1bHRzL2NvbXBhcmUuanNvbmwKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGVtcGZpbGUKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCgpmcm9tIGxlYWtkZXB0aC5pbmplY3QgaW1wb3J0IHBsYW50CgpIRVJFID0gb3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpClRPT0xTID0gKCJsZWFrZGVwdGgiLCAibGVha2NoZWNrIiwgInNwbGl0Y2hlY2siLCAibWxsZWFrIiwgImRhdGFsZWFrc19ub25lYXIiLCAic3BsaXRndWFyZCIsICJkZWVwY2hlY2tzIikKU1BMSVRfTEVBS1MgPSAoImR1cCIsICJzZXNzaW9uIiwgImhvc3QiLCAiY2FwdHVyZSIpCkxJTkVBR0VfVEYgPSAoInJlbmFtZSIsICJyb3VuZDMiLCAiY29sc3Vic2V0IiwgImNvbWJvIiwgInJlbmFtZWFsbCIpClJIT1MgPSAoMC4wLCAwLjAxLCAwLjA1KQpHUk9VUF9UT09MUyA9ICgibGVha2NoZWNrIiwgInNwbGl0Y2hlY2siLCAibWxsZWFrIiwgInNwbGl0Z3VhcmQiKQpNRVRBID0gKCJzcmNfaXAiLCAiZHN0X2lwIiwgInNyY19wb3J0IiwgImRzdF9wb3J0IiwgInByb3RvIiwgInRzIiwgImRheSIpCgoKY2xhc3MgV29ya2VyOgogICAgIiIiTG9uZy1saXZlZCB0b29sIHByb2Nlc3M7IGEgam9iIHRoYXQgZXhjZWVkcyBgYHRpbWVvdXRgYCBzZWNvbmRzIGlzIHJlY29yZGVkIGFzIGFuCiAgICBlcnJvciBhbmQgdGhlIHByb2Nlc3MgaXMgcmVzdGFydGVkLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBuYW1lOiBzdHIsIHB5dGhvbjogc3RyLCB0aW1lb3V0OiBmbG9hdCA9IDkwMCk6CiAgICAgICAgc2VsZi5uYW1lLCBzZWxmLnB5dGhvbiwgc2VsZi50aW1lb3V0ID0gbmFtZSwgcHl0aG9uLCB0aW1lb3V0CiAgICAgICAgc2VsZi5fc3RhcnQoKQoKICAgIGRlZiBfc3RhcnQoc2VsZik6CiAgICAgICAgc2VsZi5wID0gc3VicHJvY2Vzcy5Qb3Blbihbc2VsZi5weXRob24sICItSSIsIG9zLnBhdGguam9pbihIRVJFLCAiY29tcGFyZSIsIGYicnVuX3tzZWxmLm5hbWV9LnB5IildLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc3RkaW49c3VicHJvY2Vzcy5QSVBFLCBzdGRvdXQ9c3VicHJvY2Vzcy5QSVBFLCBzdGRlcnI9c3VicHJvY2Vzcy5ERVZOVUxMLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdGV4dD1UcnVlLCBidWZzaXplPTEpCgogICAgZGVmIHJ1bihzZWxmLCBqb2I6IGRpY3QpIC0+IGRpY3Q6CiAgICAgICAgaW1wb3J0IHNlbGVjdAogICAgICAgIHNlbGYucC5zdGRpbi53cml0ZShqc29uLmR1bXBzKGpvYikgKyAiXG4iKQogICAgICAgIHNlbGYucC5zdGRpbi5mbHVzaCgpCiAgICAgICAgcmVhZHksIF8sIF8gPSBzZWxlY3Quc2VsZWN0KFtzZWxmLnAuc3Rkb3V0XSwgW10sIFtdLCBzZWxmLnRpbWVvdXQpCiAgICAgICAgaWYgbm90IHJlYWR5OgogICAgICAgICAgICBzZWxmLnAua2lsbCgpCiAgICAgICAgICAgIHNlbGYucC53YWl0KCkKICAgICAgICAgICAgc2VsZi5fc3RhcnQoKQogICAgICAgICAgICByZXR1cm4geyJhcHBsaWNhYmxlIjogVHJ1ZSwgImRldGVjdGVkIjogTm9uZSwgImRldGVjdGVkX2FueSI6IE5vbmUsCiAgICAgICAgICAgICAgICAgICAgImVycm9yIjogZiJ0aW1lb3V0ID4ge3NlbGYudGltZW91dDouMGZ9IHMiLCAic2Vjb25kcyI6IHNlbGYudGltZW91dH0KICAgICAgICB3aGlsZSBUcnVlOgogICAgICAgICAgICBsaW5lID0gc2VsZi5wLnN0ZG91dC5yZWFkbGluZSgpCiAgICAgICAgICAgIGlmIG5vdCBsaW5lOgogICAgICAgICAgICAgICAgc2VsZi5fc3RhcnQoKQogICAgICAgICAgICAgICAgcmV0dXJuIHsiYXBwbGljYWJsZSI6IFRydWUsICJkZXRlY3RlZCI6IE5vbmUsICJkZXRlY3RlZF9hbnkiOiBOb25lLCAiZXJyb3IiOiAid29ya2VyIGRpZWQiLAogICAgICAgICAgICAgICAgICAgICAgICAic2Vjb25kcyI6IE5vbmV9CiAgICAgICAgICAgIGlmIGxpbmUuc3RhcnRzd2l0aCgiQEBsZWFrYmVuY2ggIik6CiAgICAgICAgICAgICAgICByZXR1cm4ganNvbi5sb2FkcyhsaW5lW2xlbigiQEBsZWFrYmVuY2ggIik6XSkKCiAgICBkZWYgY2xvc2Uoc2VsZik6CiAgICAgICAgc2VsZi5wLnN0ZGluLmNsb3NlKCkKICAgICAgICBzZWxmLnAud2FpdCh0aW1lb3V0PTMwKQoKCmRlZiBwYXJzZV9zZWVkcyhzKToKICAgIG91dCA9IFtdCiAgICBmb3IgcGFydCBpbiBzLnNwbGl0KCIsIik6CiAgICAgICAgYSwgXywgYiA9IHBhcnQucGFydGl0aW9uKCItIikKICAgICAgICBvdXQgKz0gbGlzdChyYW5nZShpbnQoYSksIGludChiKSArIDEpKSBpZiBiIGVsc2UgW2ludChhKV0KICAgIHJldHVybiBvdXQKCgpkZWYgY29uZmlncyhzZWVkcyk6CiAgICBmb3IgcyBpbiBzZWVkczoKICAgICAgICBmb3IgbGVhayBpbiBTUExJVF9MRUFLUzoKICAgICAgICAgICAgZm9yIHJobyBpbiBSSE9TOgogICAgICAgICAgICAgICAgZm9yIGNvbmQgaW4gKCJkZWZhdWx0cyIsICJmZWF0dXJlcyIsICJvcmFjbGUiKToKICAgICAgICAgICAgICAgICAgICBpZiBjb25kID09ICJvcmFjbGUiIGFuZCBsZWFrID09ICJkdXAiOgogICAgICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgICAgIHlpZWxkIHsibGVhayI6IGxlYWssICJyaG8iOiByaG8sICJzZWVkIjogcywgInRyYW5zZm9ybSI6IE5vbmUsICJjb25kaXRpb24iOiBjb25kfQogICAgICAgIGZvciB0ZiBpbiBMSU5FQUdFX1RGOgogICAgICAgICAgICBmb3IgcmhvIGluIFJIT1M6CiAgICAgICAgICAgICAgICB5aWVsZCB7ImxlYWsiOiAibGluZWFnZSIsICJyaG8iOiByaG8sICJzZWVkIjogcywgInRyYW5zZm9ybSI6IHRmLCAiY29uZGl0aW9uIjogImRlZmF1bHRzIn0KCgpkZWYga2V5KGMsIHRvb2wpOgogICAgcmV0dXJuIGYie2NbJ2xlYWsnXX18e2NbJ3JobyddfXx7Y1snc2VlZCddfXx7Y1sndHJhbnNmb3JtJ119fHtjWydjb25kaXRpb24nXX18e3Rvb2x9IgoKCmRlZiB1bml0X2NvbHVtbihsZWFrOiBzdHIsIGRmOiBwZC5EYXRhRnJhbWUpIC0+IHBkLlNlcmllczoKICAgIGlmIGxlYWsgPT0gInNlc3Npb24iOgogICAgICAgIHJldHVybiBkZlsiX3NpZCJdLmFzdHlwZShzdHIpCiAgICBpZiBsZWFrID09ICJob3N0IjoKICAgICAgICBhLCBiID0gZGZbInNyY19pcCJdLmFzdHlwZShzdHIpLCBkZlsiZHN0X2lwIl0uYXN0eXBlKHN0cikKICAgICAgICByZXR1cm4gbnAud2hlcmUoYSA8IGIsIGEgKyAifCIgKyBiLCBiICsgInwiICsgYSkKICAgIHJldHVybiBkZlsiZGF5Il0uYXN0eXBlKHN0cikgICMgY2FwdHVyZQoKCmRlZiBtYXRlcmlhbGlzZShjOiBkaWN0LCB3ZDogc3RyKSAtPiBkaWN0OgogICAgcCA9IHBsYW50KGNbImxlYWsiXSwgY1sicmhvIl0sIGNbInNlZWQiXSwgdHJhbnNmb3JtPWNbInRyYW5zZm9ybSJdIG9yICJyZW5hbWUiLCBrZWVwX3Nlc3Npb25faWQ9VHJ1ZSkKICAgIHRyLCB0ZSA9IHAudHJhaW4sIHAudGVzdAogICAgam9iID0geyJtb2RlIjogInNwbGl0IiwgImxhYmVsIjogImxhYmVsIiwgIndvcmtkaXIiOiB3ZCwgImdyb3VwIjogTm9uZX0KICAgIGlmIGNbImxlYWsiXSA9PSAibGluZWFnZSI6CiAgICAgICAgdHIgPSB0ci5kcm9wKGNvbHVtbnM9WyJfc2lkIl0sIGVycm9ycz0iaWdub3JlIikKICAgICAgICBvdGhlciA9IHAub3RoZXIuZHJvcChjb2x1bW5zPVt4IGZvciB4IGluIHAub3RoZXIuY29sdW1ucyBpZiB4ID09ICJfc2lkIl0pCiAgICAgICAgdHIudG9fY3N2KG9zLnBhdGguam9pbih3ZCwgInRyYWluLmNzdiIpLCBpbmRleD1GYWxzZSkKICAgICAgICBvdGhlci50b19jc3Yob3MucGF0aC5qb2luKHdkLCAib3RoZXIuY3N2IiksIGluZGV4PUZhbHNlKQogICAgICAgIHJldHVybiBqb2IgfCB7Im1vZGUiOiAibGluZWFnZSIsICJ0cmFpbiI6IG9zLnBhdGguam9pbih3ZCwgInRyYWluLmNzdiIpLAogICAgICAgICAgICAgICAgICAgICAgIm90aGVyIjogb3MucGF0aC5qb2luKHdkLCAib3RoZXIuY3N2IiksICJuX2xlYWtlZCI6IHAubl9sZWFrZWR9CiAgICBpZiBjWyJjb25kaXRpb24iXSA9PSAib3JhY2xlIjoKICAgICAgICB0ciA9IHRyLmFzc2lnbihfdW5pdD11bml0X2NvbHVtbihjWyJsZWFrIl0sIHRyKSkKICAgICAgICB0ZSA9IHRlLmFzc2lnbihfdW5pdD11bml0X2NvbHVtbihjWyJsZWFrIl0sIHRlKSkKICAgICAgICBqb2JbImdyb3VwIl0gPSAiX3VuaXQiCiAgICB0ciwgdGUgPSB0ci5kcm9wKGNvbHVtbnM9WyJfc2lkIl0pLCB0ZS5kcm9wKGNvbHVtbnM9WyJfc2lkIl0pCiAgICBpZiBjWyJjb25kaXRpb24iXSA9PSAiZmVhdHVyZXMiOgogICAgICAgIHRyLCB0ZSA9IHRyLmRyb3AoY29sdW1ucz1saXN0KE1FVEEpKSwgdGUuZHJvcChjb2x1bW5zPWxpc3QoTUVUQSkpCiAgICB0ci50b19jc3Yob3MucGF0aC5qb2luKHdkLCAidHJhaW4uY3N2IiksIGluZGV4PUZhbHNlKQogICAgdGUudG9fY3N2KG9zLnBhdGguam9pbih3ZCwgInRlc3QuY3N2IiksIGluZGV4PUZhbHNlKQogICAgcmV0dXJuIGpvYiB8IHsidHJhaW4iOiBvcy5wYXRoLmpvaW4od2QsICJ0cmFpbi5jc3YiKSwgInRlc3QiOiBvcy5wYXRoLmpvaW4od2QsICJ0ZXN0LmNzdiIpLCAibl9sZWFrZWQiOiBwLm5fbGVha2VkfQoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS10b29scy1kaXIiLCByZXF1aXJlZD1UcnVlLCBoZWxwPSJkaXJlY3Rvcnkgd2l0aCBvbmUgdmlydHVhbGVudiBwZXIgdG9vbCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc2VlZHMiLCByZXF1aXJlZD1UcnVlKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW91dCIsIHJlcXVpcmVkPVRydWUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tdG9vbHMiLCBkZWZhdWx0PSIsIi5qb2luKFRPT0xTKSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS10aW1lb3V0IiwgdHlwZT1mbG9hdCwgZGVmYXVsdD05MDAsIGhlbHA9InNlY29uZHMgcGVyIHRvb2wgam9iIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1sZWFrcyIsIGhlbHA9ImNvbW1hLXNlcGFyYXRlZCBzdWJzZXQgb2YgbGVhayB0eXBlcyIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcmhvcyIsIGhlbHA9ImNvbW1hLXNlcGFyYXRlZCBzdWJzZXQgb2YgcmhvIHZhbHVlcyIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY29uZGl0aW9ucyIsIGhlbHA9ImNvbW1hLXNlcGFyYXRlZCBzdWJzZXQgb2YgY29uZGl0aW9ucyIpCiAgICBhID0gYXAucGFyc2VfYXJncygpCiAgICB0b29scyA9IGEudG9vbHMuc3BsaXQoIiwiKQogICAgZG9uZSA9IHNldCgpCiAgICBpZiBvcy5wYXRoLmV4aXN0cyhhLm91dCk6CiAgICAgICAgd2l0aCBvcGVuKGEub3V0KSBhcyBmOgogICAgICAgICAgICBkb25lID0ge2tleShyLCByWyJ0b29sIl0pIGZvciByIGluIG1hcChqc29uLmxvYWRzLCBmKX0KICAgIG9zLm1ha2VkaXJzKG9zLnBhdGguZGlybmFtZShhLm91dCkgb3IgIi4iLCBleGlzdF9vaz1UcnVlKQogICAgd29ya2VycyA9IHt9CiAgICBmb3IgdCBpbiB0b29sczoKICAgICAgICBlbnYgPSB0LnNwbGl0KCJfIilbMF0KICAgICAgICBweSA9IHN5cy5leGVjdXRhYmxlIGlmIHQgPT0gImxlYWtkZXB0aCIgZWxzZSBvcy5wYXRoLmpvaW4oYS50b29sc19kaXIsIGVudiwgImJpbiIsICJweXRob24iKQogICAgICAgIHdvcmtlcnNbdF0gPSBXb3JrZXIodCwgcHksIGEudGltZW91dCkKICAgIG4gPSAwCiAgICB0cnk6CiAgICAgICAgZm9yIGMgaW4gY29uZmlncyhwYXJzZV9zZWVkcyhhLnNlZWRzKSk6CiAgICAgICAgICAgIGlmIChhLmxlYWtzIGFuZCBjWyJsZWFrIl0gbm90IGluIGEubGVha3Muc3BsaXQoIiwiKSkgb3IgXAogICAgICAgICAgICAgICAgICAgIChhLnJob3MgYW5kIGNbInJobyJdIG5vdCBpbiBbZmxvYXQoeCkgZm9yIHggaW4gYS5yaG9zLnNwbGl0KCIsIildKSBvciBcCiAgICAgICAgICAgICAgICAgICAgKGEuY29uZGl0aW9ucyBhbmQgY1siY29uZGl0aW9uIl0gbm90IGluIGEuY29uZGl0aW9ucy5zcGxpdCgiLCIpKToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHRvZG8gPSBbdCBmb3IgdCBpbiB0b29scyBpZiBrZXkoYywgdCkgbm90IGluIGRvbmVdCiAgICAgICAgICAgIGlmIG5vdCB0b2RvOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgd2l0aCB0ZW1wZmlsZS5UZW1wb3JhcnlEaXJlY3RvcnkoKSBhcyB3ZDoKICAgICAgICAgICAgICAgIGpvYiA9IG1hdGVyaWFsaXNlKGMsIHdkKQogICAgICAgICAgICAgICAgZm9yIHQgaW4gdG9kbzoKICAgICAgICAgICAgICAgICAgICBqID0gZGljdChqb2IpCiAgICAgICAgICAgICAgICAgICAgaWYgai5nZXQoImdyb3VwIikgYW5kIHQgbm90IGluIEdST1VQX1RPT0xTOgogICAgICAgICAgICAgICAgICAgICAgICBqWyJncm91cCJdID0gTm9uZSAgIyB0b29sIGhhcyBubyBncm91cCBvcHRpb246IHJ1biBhcyBkZWZhdWx0cyAoKyBleHRyYSBjb2x1bW4pCiAgICAgICAgICAgICAgICAgICAgciA9IHdvcmtlcnNbdF0ucnVuKGopCiAgICAgICAgICAgICAgICAgICAgcm93ID0geyoqYywgInRvb2wiOiB0LCAibl9sZWFrZWQiOiBqb2JbIm5fbGVha2VkIl0sICJncm91cF9wYXNzZWQiOiBib29sKGouZ2V0KCJncm91cCIpKSwgKipyfQogICAgICAgICAgICAgICAgICAgIHJvdy5wb3AoInRyYWNlIiwgTm9uZSkKICAgICAgICAgICAgICAgICAgICB3aXRoIG9wZW4oYS5vdXQsICJhIikgYXMgZjoKICAgICAgICAgICAgICAgICAgICAgICAgZi53cml0ZShqc29uLmR1bXBzKHJvdywgZGVmYXVsdD1zdHIpICsgIlxuIikKICAgICAgICAgICAgICAgICAgICBuICs9IDEKICAgIGZpbmFsbHk6CiAgICAgICAgZm9yIHcgaW4gd29ya2Vycy52YWx1ZXMoKToKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgdy5jbG9zZSgpCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246ICAjIG5vcWE6IEJMRTAwMQogICAgICAgICAgICAgICAgcGFzcwogICAgcHJpbnQoZiJ7bn0gbmV3IHJlc3VsdHMgd3JpdHRlbiB0byB7YS5vdXR9IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=='}, 'compare/run_leakdepth.py': {'sha256': 'e64b7869a941989c2b94d35c56f1143c27a53f2414c7791ae1efd02416db1eb4', 'b64': 'IiIibGVha2RlcHRoOiBhdWRpdF9zcGxpdCAoc3BsaXQgbW9kZSkgb3IgbGluZWFnZTsgZGV0ZWN0ZWQgPSBhbnkgV0FSTiBvciBFUlJPUiBmaW5kaW5nLiIiIgppbXBvcnQgb3MKaW1wb3J0IHN5cwoKaW1wb3J0IHBhbmRhcyBhcyBwZAoKc3lzLnBhdGguaW5zZXJ0KDAsIG9zLnBhdGguZGlybmFtZShfX2ZpbGVfXykpCmZyb20gX2xvb3AgaW1wb3J0IHNlcnZlCgpmcm9tIGxlYWtkZXB0aCBpbXBvcnQgQ29sdW1uTWFwLCBGbG93VGFibGUsIGF1ZGl0X3NwbGl0LCBjb25jYXQsIGxpbmVhZ2UKClJPTEVTID0geyJzcmMiOiAic3JjX2lwIiwgImRzdCI6ICJkc3RfaXAiLCAic3BvcnQiOiAic3JjX3BvcnQiLCAiZHBvcnQiOiAiZHN0X3BvcnQiLCAicHJvdG8iOiAicHJvdG8iLAogICAgICAgICAidGltZSI6ICJ0cyIsICJjYXB0dXJlIjogImRheSJ9CgoKZGVmIGhhbmRsZShqb2IpOgogICAgdHIgPSBwZC5yZWFkX2Nzdihqb2JbInRyYWluIl0sIGxvd19tZW1vcnk9RmFsc2UpCiAgICBpZiBqb2JbIm1vZGUiXSA9PSAic3BsaXQiOgogICAgICAgIHRlID0gcGQucmVhZF9jc3Yoam9iWyJ0ZXN0Il0sIGxvd19tZW1vcnk9RmFsc2UpCiAgICAgICAgcm9sZXMgPSB7azogdiBmb3IgaywgdiBpbiBST0xFUy5pdGVtcygpIGlmIHYgaW4gdHIuY29sdW1uc30KICAgICAgICBjb2xzID0gQ29sdW1uTWFwKGxhYmVsPWpvYlsibGFiZWwiXSwgZXhjbHVkZT10dXBsZShjIGZvciBjIGluIChqb2IuZ2V0KCJncm91cCIpLCkgaWYgYyksICoqcm9sZXMpCiAgICAgICAgdCwgYSwgYiA9IGNvbmNhdChGbG93VGFibGUodHIsIGNvbHMpLCBGbG93VGFibGUodGUsIGNvbHMpKQogICAgICAgIHJlcCA9IGF1ZGl0X3NwbGl0KHQsIGEsIGIsIGNyb3NzX2NhcHR1cmU9VHJ1ZSkKICAgIGVsc2U6CiAgICAgICAgcmVwID0gbGluZWFnZSh0ciwgcGQucmVhZF9jc3Yoam9iWyJvdGhlciJdLCBsb3dfbWVtb3J5PUZhbHNlKSkKICAgIGZpcmVkID0gc29ydGVkKHtmLmlkIGZvciBmIGluIHJlcC5maW5kaW5ncyBpZiBmLnNldmVyaXR5IGluICgiV0FSTiIsICJFUlJPUiIpfSkKICAgIHJldHVybiB7ImFwcGxpY2FibGUiOiBUcnVlLCAiZGV0ZWN0ZWQiOiBib29sKGZpcmVkKSwgImRldGVjdGVkX2FueSI6IGJvb2woZmlyZWQpLCAiZGV0YWlsIjogeyJmaXJlZCI6IGZpcmVkfSwKICAgICAgICAgICAgImVycm9yIjogTm9uZX0KCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgc2VydmUoaGFuZGxlKQo='}, 'compare/_loop.py': {'sha256': '1ceee163692624dfa9caa21d71cd71fbd83b3e283d3ffb80c46acbe38cbdd94c', 'b64': 'IiIiSlNPTi1saW5lcyB3b3JrZXIgbG9vcCBzaGFyZWQgYnkgdGhlIHRvb2wgcnVubmVycyAob25lIGpvYiBwZXIgc3RkaW4gbGluZSkuIiIiCmltcG9ydCBjb250ZXh0bGliCmltcG9ydCBqc29uCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKaW1wb3J0IHRyYWNlYmFjawppbXBvcnQgd2FybmluZ3MKCk1BUksgPSAiQEBsZWFrYmVuY2ggIgoKCmRlZiBzZXJ2ZShoYW5kbGUpOgogICAgd2FybmluZ3MuZmlsdGVyd2FybmluZ3MoImlnbm9yZSIpCiAgICBmb3IgbGluZSBpbiBzeXMuc3RkaW46CiAgICAgICAgbGluZSA9IGxpbmUuc3RyaXAoKQogICAgICAgIGlmIG5vdCBsaW5lOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGpvYiA9IGpzb24ubG9hZHMobGluZSkKICAgICAgICB0MCA9IHRpbWUudGltZSgpCiAgICAgICAgdHJ5OgogICAgICAgICAgICB3aXRoIGNvbnRleHRsaWIucmVkaXJlY3Rfc3Rkb3V0KHN5cy5zdGRlcnIpOiAgIyB0b29scyBwcmludCByZXBvcnRzOyBrZWVwIHRoZSBjaGFubmVsIGNsZWFuCiAgICAgICAgICAgICAgICBvdXQgPSBoYW5kbGUoam9iKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZTogICMgbm9xYTogQkxFMDAxCiAgICAgICAgICAgIG91dCA9IHsiYXBwbGljYWJsZSI6IFRydWUsICJkZXRlY3RlZCI6IE5vbmUsICJkZXRlY3RlZF9hbnkiOiBOb25lLAogICAgICAgICAgICAgICAgICAgImVycm9yIjogZiJ7dHlwZShlKS5fX25hbWVfX306IHtlfSJbOjMwMF0sICJ0cmFjZSI6IHRyYWNlYmFjay5mb3JtYXRfZXhjKClbLTYwMDpdfQogICAgICAgIG91dFsic2Vjb25kcyJdID0gcm91bmQodGltZS50aW1lKCkgLSB0MCwgMikKICAgICAgICBzeXMuX19zdGRvdXRfXy53cml0ZShNQVJLICsganNvbi5kdW1wcyhvdXQsIGRlZmF1bHQ9c3RyKSArICJcbiIpCiAgICAgICAgc3lzLl9fc3Rkb3V0X18uZmx1c2goKQo='}}
LD010_ROWS = [{"leak":"dup","rho":0.0,"seed":100,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"dup","rho":0.0,"seed":100,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"dup","rho":0.01,"seed":100,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.01,"seed":100,"transform":None,"condition":"features","detected":True,"error":None},{"leak":"dup","rho":0.05,"seed":100,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.05,"seed":100,"transform":None,"condition":"features","detected":True,"error":None},{"leak":"session","rho":0.0,"seed":100,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"session","rho":0.0,"seed":100,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.0,"seed":100,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"session","rho":0.01,"seed":100,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"session","rho":0.01,"seed":100,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.01,"seed":100,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"session","rho":0.05,"seed":100,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"session","rho":0.05,"seed":100,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.05,"seed":100,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"host","rho":0.0,"seed":100,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"host","rho":0.0,"seed":100,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.0,"seed":100,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"host","rho":0.01,"seed":100,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"host","rho":0.01,"seed":100,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.01,"seed":100,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"host","rho":0.05,"seed":100,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"host","rho":0.05,"seed":100,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.05,"seed":100,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"capture","rho":0.0,"seed":100,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"capture","rho":0.0,"seed":100,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.0,"seed":100,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"capture","rho":0.01,"seed":100,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"capture","rho":0.01,"seed":100,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.01,"seed":100,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"capture","rho":0.05,"seed":100,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"capture","rho":0.05,"seed":100,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.05,"seed":100,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":100,"transform":"rename","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":100,"transform":"rename","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":100,"transform":"rename","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":100,"transform":"round3","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":100,"transform":"round3","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":100,"transform":"round3","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":100,"transform":"colsubset","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":100,"transform":"colsubset","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":100,"transform":"colsubset","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":100,"transform":"combo","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":100,"transform":"combo","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":100,"transform":"combo","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":100,"transform":"renameall","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":100,"transform":"renameall","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":100,"transform":"renameall","condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.0,"seed":101,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"dup","rho":0.0,"seed":101,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"dup","rho":0.01,"seed":101,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.01,"seed":101,"transform":None,"condition":"features","detected":True,"error":None},{"leak":"dup","rho":0.05,"seed":101,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.05,"seed":101,"transform":None,"condition":"features","detected":True,"error":None},{"leak":"session","rho":0.0,"seed":101,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"session","rho":0.0,"seed":101,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.0,"seed":101,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"session","rho":0.01,"seed":101,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"session","rho":0.01,"seed":101,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.01,"seed":101,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"session","rho":0.05,"seed":101,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"session","rho":0.05,"seed":101,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.05,"seed":101,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"host","rho":0.0,"seed":101,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"host","rho":0.0,"seed":101,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.0,"seed":101,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"host","rho":0.01,"seed":101,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"host","rho":0.01,"seed":101,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.01,"seed":101,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"host","rho":0.05,"seed":101,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"host","rho":0.05,"seed":101,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.05,"seed":101,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"capture","rho":0.0,"seed":101,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"capture","rho":0.0,"seed":101,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.0,"seed":101,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"capture","rho":0.01,"seed":101,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"capture","rho":0.01,"seed":101,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.01,"seed":101,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"capture","rho":0.05,"seed":101,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"capture","rho":0.05,"seed":101,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.05,"seed":101,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":101,"transform":"rename","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":101,"transform":"rename","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":101,"transform":"rename","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":101,"transform":"round3","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":101,"transform":"round3","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":101,"transform":"round3","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":101,"transform":"colsubset","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":101,"transform":"colsubset","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":101,"transform":"colsubset","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":101,"transform":"combo","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":101,"transform":"combo","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":101,"transform":"combo","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":101,"transform":"renameall","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":101,"transform":"renameall","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":101,"transform":"renameall","condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.0,"seed":102,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"dup","rho":0.0,"seed":102,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"dup","rho":0.01,"seed":102,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.01,"seed":102,"transform":None,"condition":"features","detected":True,"error":None},{"leak":"dup","rho":0.05,"seed":102,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.05,"seed":102,"transform":None,"condition":"features","detected":True,"error":None},{"leak":"session","rho":0.0,"seed":102,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"session","rho":0.0,"seed":102,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.0,"seed":102,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"session","rho":0.01,"seed":102,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"session","rho":0.01,"seed":102,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.01,"seed":102,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"session","rho":0.05,"seed":102,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"session","rho":0.05,"seed":102,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.05,"seed":102,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"host","rho":0.0,"seed":102,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"host","rho":0.0,"seed":102,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.0,"seed":102,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"host","rho":0.01,"seed":102,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"host","rho":0.01,"seed":102,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.01,"seed":102,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"host","rho":0.05,"seed":102,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"host","rho":0.05,"seed":102,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.05,"seed":102,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"capture","rho":0.0,"seed":102,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"capture","rho":0.0,"seed":102,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.0,"seed":102,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"capture","rho":0.01,"seed":102,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"capture","rho":0.01,"seed":102,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.01,"seed":102,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"capture","rho":0.05,"seed":102,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"capture","rho":0.05,"seed":102,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.05,"seed":102,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":102,"transform":"rename","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":102,"transform":"rename","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":102,"transform":"rename","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":102,"transform":"round3","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":102,"transform":"round3","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":102,"transform":"round3","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":102,"transform":"colsubset","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":102,"transform":"colsubset","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":102,"transform":"colsubset","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":102,"transform":"combo","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":102,"transform":"combo","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":102,"transform":"combo","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":102,"transform":"renameall","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":102,"transform":"renameall","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":102,"transform":"renameall","condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.0,"seed":103,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"dup","rho":0.0,"seed":103,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"dup","rho":0.01,"seed":103,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.01,"seed":103,"transform":None,"condition":"features","detected":True,"error":None},{"leak":"dup","rho":0.05,"seed":103,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.05,"seed":103,"transform":None,"condition":"features","detected":True,"error":None},{"leak":"session","rho":0.0,"seed":103,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"session","rho":0.0,"seed":103,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.0,"seed":103,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"session","rho":0.01,"seed":103,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"session","rho":0.01,"seed":103,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.01,"seed":103,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"session","rho":0.05,"seed":103,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"session","rho":0.05,"seed":103,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.05,"seed":103,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"host","rho":0.0,"seed":103,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"host","rho":0.0,"seed":103,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.0,"seed":103,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"host","rho":0.01,"seed":103,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"host","rho":0.01,"seed":103,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.01,"seed":103,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"host","rho":0.05,"seed":103,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"host","rho":0.05,"seed":103,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.05,"seed":103,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"capture","rho":0.0,"seed":103,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"capture","rho":0.0,"seed":103,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.0,"seed":103,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"capture","rho":0.01,"seed":103,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"capture","rho":0.01,"seed":103,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.01,"seed":103,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"capture","rho":0.05,"seed":103,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"capture","rho":0.05,"seed":103,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.05,"seed":103,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":103,"transform":"rename","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":103,"transform":"rename","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":103,"transform":"rename","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":103,"transform":"round3","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":103,"transform":"round3","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":103,"transform":"round3","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":103,"transform":"colsubset","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":103,"transform":"colsubset","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":103,"transform":"colsubset","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":103,"transform":"combo","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":103,"transform":"combo","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":103,"transform":"combo","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":103,"transform":"renameall","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":103,"transform":"renameall","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.05,"seed":103,"transform":"renameall","condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.0,"seed":104,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"dup","rho":0.0,"seed":104,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"dup","rho":0.01,"seed":104,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.01,"seed":104,"transform":None,"condition":"features","detected":True,"error":None},{"leak":"dup","rho":0.05,"seed":104,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.05,"seed":104,"transform":None,"condition":"features","detected":True,"error":None},{"leak":"session","rho":0.0,"seed":104,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"session","rho":0.0,"seed":104,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.0,"seed":104,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"session","rho":0.01,"seed":104,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"session","rho":0.01,"seed":104,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.01,"seed":104,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"session","rho":0.05,"seed":104,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"session","rho":0.05,"seed":104,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.05,"seed":104,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"host","rho":0.0,"seed":104,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"host","rho":0.0,"seed":104,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.0,"seed":104,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"host","rho":0.01,"seed":104,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"host","rho":0.01,"seed":104,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.01,"seed":104,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"host","rho":0.05,"seed":104,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"host","rho":0.05,"seed":104,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.05,"seed":104,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"capture","rho":0.0,"seed":104,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"capture","rho":0.0,"seed":104,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.0,"seed":104,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"capture","rho":0.01,"seed":104,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"capture","rho":0.01,"seed":104,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.01,"seed":104,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"capture","rho":0.05,"seed":104,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"capture","rho":0.05,"seed":104,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.05,"seed":104,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":104,"transform":"rename","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":104,"transform":"rename","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":104,"transform":"rename","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":104,"transform":"round3","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":104,"transform":"round3","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":104,"transform":"round3","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":104,"transform":"colsubset","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":104,"transform":"colsubset","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":104,"transform":"colsubset","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":104,"transform":"combo","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":104,"transform":"combo","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":104,"transform":"combo","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":104,"transform":"renameall","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":104,"transform":"renameall","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":104,"transform":"renameall","condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.0,"seed":105,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"dup","rho":0.0,"seed":105,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"dup","rho":0.01,"seed":105,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.01,"seed":105,"transform":None,"condition":"features","detected":True,"error":None},{"leak":"dup","rho":0.05,"seed":105,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.05,"seed":105,"transform":None,"condition":"features","detected":True,"error":None},{"leak":"session","rho":0.0,"seed":105,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"session","rho":0.0,"seed":105,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.0,"seed":105,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"session","rho":0.01,"seed":105,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"session","rho":0.01,"seed":105,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.01,"seed":105,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"session","rho":0.05,"seed":105,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"session","rho":0.05,"seed":105,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.05,"seed":105,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"host","rho":0.0,"seed":105,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"host","rho":0.0,"seed":105,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.0,"seed":105,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"host","rho":0.01,"seed":105,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"host","rho":0.01,"seed":105,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.01,"seed":105,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"host","rho":0.05,"seed":105,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"host","rho":0.05,"seed":105,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.05,"seed":105,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"capture","rho":0.0,"seed":105,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"capture","rho":0.0,"seed":105,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.0,"seed":105,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"capture","rho":0.01,"seed":105,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"capture","rho":0.01,"seed":105,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.01,"seed":105,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"capture","rho":0.05,"seed":105,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"capture","rho":0.05,"seed":105,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.05,"seed":105,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":105,"transform":"rename","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":105,"transform":"rename","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":105,"transform":"rename","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":105,"transform":"round3","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":105,"transform":"round3","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":105,"transform":"round3","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":105,"transform":"colsubset","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":105,"transform":"colsubset","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":105,"transform":"colsubset","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":105,"transform":"combo","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":105,"transform":"combo","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":105,"transform":"combo","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":105,"transform":"renameall","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":105,"transform":"renameall","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":105,"transform":"renameall","condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.0,"seed":106,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"dup","rho":0.0,"seed":106,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"dup","rho":0.01,"seed":106,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.01,"seed":106,"transform":None,"condition":"features","detected":True,"error":None},{"leak":"dup","rho":0.05,"seed":106,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.05,"seed":106,"transform":None,"condition":"features","detected":True,"error":None},{"leak":"session","rho":0.0,"seed":106,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"session","rho":0.0,"seed":106,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.0,"seed":106,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"session","rho":0.01,"seed":106,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"session","rho":0.01,"seed":106,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.01,"seed":106,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"session","rho":0.05,"seed":106,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"session","rho":0.05,"seed":106,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.05,"seed":106,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"host","rho":0.0,"seed":106,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"host","rho":0.0,"seed":106,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.0,"seed":106,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"host","rho":0.01,"seed":106,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"host","rho":0.01,"seed":106,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.01,"seed":106,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"host","rho":0.05,"seed":106,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"host","rho":0.05,"seed":106,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.05,"seed":106,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"capture","rho":0.0,"seed":106,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"capture","rho":0.0,"seed":106,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.0,"seed":106,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"capture","rho":0.01,"seed":106,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"capture","rho":0.01,"seed":106,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.01,"seed":106,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"capture","rho":0.05,"seed":106,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"capture","rho":0.05,"seed":106,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.05,"seed":106,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":106,"transform":"rename","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":106,"transform":"rename","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.05,"seed":106,"transform":"rename","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":106,"transform":"round3","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":106,"transform":"round3","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.05,"seed":106,"transform":"round3","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":106,"transform":"colsubset","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":106,"transform":"colsubset","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.05,"seed":106,"transform":"colsubset","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":106,"transform":"combo","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":106,"transform":"combo","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.05,"seed":106,"transform":"combo","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":106,"transform":"renameall","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":106,"transform":"renameall","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.05,"seed":106,"transform":"renameall","condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.0,"seed":107,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"dup","rho":0.0,"seed":107,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"dup","rho":0.01,"seed":107,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.01,"seed":107,"transform":None,"condition":"features","detected":True,"error":None},{"leak":"dup","rho":0.05,"seed":107,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.05,"seed":107,"transform":None,"condition":"features","detected":True,"error":None},{"leak":"session","rho":0.0,"seed":107,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"session","rho":0.0,"seed":107,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.0,"seed":107,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"session","rho":0.01,"seed":107,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"session","rho":0.01,"seed":107,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.01,"seed":107,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"session","rho":0.05,"seed":107,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"session","rho":0.05,"seed":107,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.05,"seed":107,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"host","rho":0.0,"seed":107,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"host","rho":0.0,"seed":107,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.0,"seed":107,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"host","rho":0.01,"seed":107,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"host","rho":0.01,"seed":107,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.01,"seed":107,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"host","rho":0.05,"seed":107,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"host","rho":0.05,"seed":107,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.05,"seed":107,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"capture","rho":0.0,"seed":107,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"capture","rho":0.0,"seed":107,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.0,"seed":107,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"capture","rho":0.01,"seed":107,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"capture","rho":0.01,"seed":107,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.01,"seed":107,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"capture","rho":0.05,"seed":107,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"capture","rho":0.05,"seed":107,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.05,"seed":107,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":107,"transform":"rename","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":107,"transform":"rename","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":107,"transform":"rename","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":107,"transform":"round3","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":107,"transform":"round3","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":107,"transform":"round3","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":107,"transform":"colsubset","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":107,"transform":"colsubset","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":107,"transform":"colsubset","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":107,"transform":"combo","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":107,"transform":"combo","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":107,"transform":"combo","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":107,"transform":"renameall","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":107,"transform":"renameall","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":107,"transform":"renameall","condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.0,"seed":108,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"dup","rho":0.0,"seed":108,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"dup","rho":0.01,"seed":108,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.01,"seed":108,"transform":None,"condition":"features","detected":True,"error":None},{"leak":"dup","rho":0.05,"seed":108,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.05,"seed":108,"transform":None,"condition":"features","detected":True,"error":None},{"leak":"session","rho":0.0,"seed":108,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"session","rho":0.0,"seed":108,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.0,"seed":108,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"session","rho":0.01,"seed":108,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"session","rho":0.01,"seed":108,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.01,"seed":108,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"session","rho":0.05,"seed":108,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"session","rho":0.05,"seed":108,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.05,"seed":108,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"host","rho":0.0,"seed":108,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"host","rho":0.0,"seed":108,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.0,"seed":108,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"host","rho":0.01,"seed":108,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"host","rho":0.01,"seed":108,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.01,"seed":108,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"host","rho":0.05,"seed":108,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"host","rho":0.05,"seed":108,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.05,"seed":108,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"capture","rho":0.0,"seed":108,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"capture","rho":0.0,"seed":108,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.0,"seed":108,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"capture","rho":0.01,"seed":108,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"capture","rho":0.01,"seed":108,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.01,"seed":108,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"capture","rho":0.05,"seed":108,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"capture","rho":0.05,"seed":108,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.05,"seed":108,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":108,"transform":"rename","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":108,"transform":"rename","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.05,"seed":108,"transform":"rename","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":108,"transform":"round3","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":108,"transform":"round3","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":108,"transform":"round3","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":108,"transform":"colsubset","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":108,"transform":"colsubset","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":108,"transform":"colsubset","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":108,"transform":"combo","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":108,"transform":"combo","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.05,"seed":108,"transform":"combo","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":108,"transform":"renameall","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":108,"transform":"renameall","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.05,"seed":108,"transform":"renameall","condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.0,"seed":109,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"dup","rho":0.0,"seed":109,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"dup","rho":0.01,"seed":109,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.01,"seed":109,"transform":None,"condition":"features","detected":True,"error":None},{"leak":"dup","rho":0.05,"seed":109,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"dup","rho":0.05,"seed":109,"transform":None,"condition":"features","detected":True,"error":None},{"leak":"session","rho":0.0,"seed":109,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"session","rho":0.0,"seed":109,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.0,"seed":109,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"session","rho":0.01,"seed":109,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"session","rho":0.01,"seed":109,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.01,"seed":109,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"session","rho":0.05,"seed":109,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"session","rho":0.05,"seed":109,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"session","rho":0.05,"seed":109,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"host","rho":0.0,"seed":109,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"host","rho":0.0,"seed":109,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.0,"seed":109,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"host","rho":0.01,"seed":109,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"host","rho":0.01,"seed":109,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.01,"seed":109,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"host","rho":0.05,"seed":109,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"host","rho":0.05,"seed":109,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"host","rho":0.05,"seed":109,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"capture","rho":0.0,"seed":109,"transform":None,"condition":"defaults","detected":False,"error":None},{"leak":"capture","rho":0.0,"seed":109,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.0,"seed":109,"transform":None,"condition":"oracle","detected":False,"error":None},{"leak":"capture","rho":0.01,"seed":109,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"capture","rho":0.01,"seed":109,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.01,"seed":109,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"capture","rho":0.05,"seed":109,"transform":None,"condition":"defaults","detected":True,"error":None},{"leak":"capture","rho":0.05,"seed":109,"transform":None,"condition":"features","detected":False,"error":None},{"leak":"capture","rho":0.05,"seed":109,"transform":None,"condition":"oracle","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":109,"transform":"rename","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":109,"transform":"rename","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.05,"seed":109,"transform":"rename","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":109,"transform":"round3","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":109,"transform":"round3","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.05,"seed":109,"transform":"round3","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":109,"transform":"colsubset","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":109,"transform":"colsubset","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.05,"seed":109,"transform":"colsubset","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":109,"transform":"combo","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":109,"transform":"combo","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.05,"seed":109,"transform":"combo","condition":"defaults","detected":True,"error":None},{"leak":"lineage","rho":0.0,"seed":109,"transform":"renameall","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.01,"seed":109,"transform":"renameall","condition":"defaults","detected":False,"error":None},{"leak":"lineage","rho":0.05,"seed":109,"transform":"renameall","condition":"defaults","detected":True,"error":None}]
w = f"{WD}/leakdepth-0.1.1-py3-none-any.whl"
open(w, "wb").write(base64.b64decode(WHEEL_B64))
assert hashlib.sha256(open(w, "rb").read()).hexdigest() == WHEEL_SHA, "0.1.1 wheel hash mismatch"
if not LOCAL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", w, "pyarrow", "huggingface_hub", "matplotlib"], check=True)
for name, v in BENCH.items():                           # bench/compare.py, bench/compare/run_leakdepth.py, _loop.py
    p = f"{WD}/bench/{name}"; os.makedirs(os.path.dirname(p), exist_ok=True)
    b = base64.b64decode(v["b64"]); assert hashlib.sha256(b).hexdigest() == v["sha256"], name
    open(p, "wb").write(b)
import numpy as np, pandas as pd, scipy, leakdepth as L
LL = importlib.import_module("leakdepth.lineage")
from leakdepth.metrics import wilson
assert L.__version__ == "0.1.1", L.__version__
print("leakdepth", L.__version__, "| numpy", np.__version__, "| pandas", pd.__version__, "| CPUs", os.cpu_count(),
      "| RAM GB", round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1))
T0 = time.time()
MODES = {"a10": {"matcher": "a10", "info_margin": 20.0}, "frozen": {"matcher": "frozen"}}

def done(name):
    return os.path.exists(f"{OUT}/{name}")

def jdump(obj, name):
    tmp = f"{OUT}/{name}.tmp"
    json.dump(obj, open(tmp, "w"), default=lambda o: o.item() if hasattr(o, "item") else str(o))
    os.replace(tmp, f"{OUT}/{name}")

def jload(name):
    return json.load(open(f"{OUT}/{name}"))

def cont(d):
    return [(c["direction"], c["matched"], c["sampled_rows"], round(c["containment"], 6), round(c["ci_hi"], 6))
            for c in d["results"]["containment"]]

FILES6 = {"ahuja": "ahuja__dataset_sdn.csv", "karmous": "karmous__SDN_DDoS_Dataset.csv",
          "snt": "snt__SNT_dataset.csv", "ukm": "ukmddosflow__flow_features.csv"}

def read6(nm):
    return pd.read_csv(f"{RAW6}/{FILES6[nm]}", low_memory=False)


In [ ]:
# 2. E (gate): 0.1.1 reproduces the NB3 counts on the Ahuja-SNT control
if done("E.json"):
    E = jload("E.json")
else:
    a, b = read6("ahuja"), read6("snt")
    got = {m: [c[1] for c in cont(L.lineage(a, b, "Ahuja", "snt", seed=0, **kw).to_dict())] for m, kw in MODES.items()}
    expect = {"frozen": [2681, 6993], "a10": [0, 0]}
    E = {"got": got, "expected_nb3": expect, "ok": got == expect or bool(LOCAL)}
    jdump(E, "E.json"); del a, b; gc.collect()
print("E gate:", "PASS" if E["ok"] else "FAIL -> pre-registered STOP", E["got"])


In [ ]:
# 3. N: new negative controls (not used to design A10); exact-overlap gate; A10 criterion C <= 0.01 both directions
HF = {"cic2018": ("c01dsnap/CIC-IDS2018", "236baa10495af451de38485123a90f60e7632bbd",
                  ["Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv", "Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv"]),
      "insdn": ("Sharukesh/INSDN", "f59330c8ff0f1b231b382e05fc41b1e8fb0df6b4", ["Dataset.csv"]),
      "toniot": ("codymlewis/TON_IoT_network", "b953d9388d2db3bc392ea7150476e4f219a2a1cb", ["train_test_network.csv"]),
      "nslkdd": ("Mireu-Lab/NSL-KDD", "e755a229b5c65c7dd6a3714fd20b25558713543c", ["train.csv"]),
      "cic2017": ("rdpahalavan/CIC-IDS2017", None, ["Network-Flows/CICIDS_Flow.parquet"]),
      "unsw": ("Mouwiya/UNSW-NB15", None, ["data/train-00000-of-00002.parquet", "data/train-00001-of-00002.parquet"])}
MIRROR_SHA = {"Network-Flows/CICIDS_Flow.parquet": "e64f6adfbcb7f96b018cf36c731be64ef4de3353ccd7340b190590734a0dda35",
              "data/train-00000-of-00002.parquet": "2aada2a26d061111f4e8fb84e716f5f11264fee71abe04697d42cb89e488d047",
              "data/train-00001-of-00002.parquet": "7c6699ae967567977dee9e9193543b515255f4e1671ca79bf9ae65e7866ffef1"}
FILE_LOG = {}

def load(name):
    if LOCAL:
        return pd.read_parquet(f"{BASE}/smoke_{name}.parquet")
    from huggingface_hub import hf_hub_download
    repo, rev, files = HF[name]
    parts = []
    for f in files:
        p = hf_hub_download(repo, f, repo_type="dataset", revision=rev, local_dir=f"/content/data/{repo.replace('/', '__')}")
        h = hashlib.sha256(open(p, "rb").read()).hexdigest()
        FILE_LOG[f"{repo}/{f}"] = {"revision": rev, "sha256": h, "bytes": os.path.getsize(p)}
        if f in MIRROR_SHA:
            assert h == MIRROR_SHA[f], f"mirror changed: {f}"
        parts.append(pd.read_parquet(p) if f.endswith(".parquet") else pd.read_csv(p, low_memory=False))
    return pd.concat(parts, ignore_index=True)

def exact_overlap(a, b):
    """Share of rows of each side whose values on the columns shared by normalised name occur in the other side."""
    na = {LL.norm_name(c): c for c in a.columns}; nb = {LL.norm_name(c): c for c in b.columns}
    shared = sorted(set(na) & set(nb))
    if len(shared) < 4:
        return {"shared": shared, "checked": False}
    def keys(df, cols):
        parts = []
        for c in cols:
            v = pd.to_numeric(df[c], errors="coerce")
            parts.append(np.where(v.notna(), v.round(9).astype(str), df[c].astype(str).str.strip().str.lower()))
        return pd.util.hash_pandas_object(pd.DataFrame(dict(enumerate(parts))), index=False).to_numpy()
    ka, kb = keys(a, [na[s] for s in shared]), keys(b, [nb[s] for s in shared])
    sa, sb = set(kb.tolist()), set(ka.tolist())
    oa, ob = float(np.mean([k in sa for k in ka])), float(np.mean([k in sb for k in kb]))
    return {"shared": shared, "checked": True, "a_in_b": oa, "b_in_a": ob}

PAIRS = [("N1", "cic2017", "cic2018"), ("N2", "insdn", "cic2017"), ("N3", "insdn", "cic2018"),
         ("N4", "toniot", "unsw"), ("N5", "nslkdd", "unsw")]
hirsi = sorted(glob.glob(f"{RAW6}/*hirsi*") + glob.glob(f"{RAW6}/*SDN-DDoS*") + glob.glob(f"{RAW6}/*sdn_ddos_traffic*"))
if hirsi:
    PAIRS += [(f"N6-{o}", "hirsi", o) for o in ("ahuja", "karmous", "snt", "ukm")]
print("pairs:", [p[0] for p in PAIRS], "| SDN-DDoS Traffic file:", hirsi[:1] or "not found in " + RAW6)
CACHE = {}

def get(name):
    if name not in CACHE:
        if name == "hirsi":
            CACHE[name] = pd.read_csv(hirsi[0], low_memory=False)
        elif name in FILES6:
            CACHE[name] = read6(name)
        else:
            CACHE[name] = load(name)
        if SMOKE:
            CACHE[name] = CACHE[name].sample(n=min(20000, len(CACHE[name])), random_state=0).reset_index(drop=True)
    return CACHE[name]

if not E["ok"]:
    print("E failed: skipped (pre-registered stop)")
else:
    for pid, x, y in PAIRS:
        fn = f"N_{pid}.json"
        if not done(fn):
            a, b = get(x), get(y)
            t0 = time.time(); gate = exact_overlap(a, b)
            res = {"pair": pid, "a": x, "b": y, "rows": [len(a), len(b)], "gate": gate, "files": dict(FILE_LOG)}
            for m, kw in MODES.items():
                t1 = time.time(); d = L.lineage(a, b, x, y, seed=0, **kw).to_dict()
                res[m] = {"containment": cont(d), "aligned": len(d["results"].get("alignment", [])),
                          "rejected_low_info": [c.get("rejected_low_info") for c in d["results"]["containment"]],
                          "seconds": round(time.time() - t1, 1)}
            res["seconds_total"] = round(time.time() - t0, 1)
            jdump(res, fn)
        r = jload(fn)
        g = r["gate"]
        print(pid, x, "vs", y, "| shared names", len(g["shared"]), "| exact overlap",
              (round(g["a_in_b"], 5), round(g["b_in_a"], 5)) if g["checked"] else "not checked",
              "| A10", [(c[1], c[2], c[3]) for c in r["a10"]["containment"]], "| frozen", [(c[1], c[2], c[3]) for c in r["frozen"]["containment"]])
        for nm in list(CACHE):                          # free memory between pairs that do not reuse a corpus
            if all(nm not in (p[1], p[2]) for p in PAIRS[PAIRS.index((pid, x, y)) + 1:]):
                del CACHE[nm]
        gc.collect()


In [ ]:
# 4. T: threshold-free lineage measure (planted rows identified by replaying the generator)
from leakdepth.inject import corpus, transform_corpus, plant, TRANSFORMS
TFS = list(TRANSFORMS) + ["renameall"]
THRESH = [0.001, 0.0025, 0.005, 0.01, 0.02, 0.05]
TF_FILE = f"{OUT}/T_threshold_free.jsonl"

def replay(rho, seed, tf):
    train, test, g, info, _ = corpus(seed, 1200)
    r = g.r
    fresh_tr, fresh_te, *_ = corpus(seed + 7919, 1200)
    fresh = pd.concat([fresh_tr, fresh_te], ignore_index=True)
    kb = int(round(rho * len(test)))
    src = train.iloc[r.choice(len(train), kb, replace=False)] if kb else train.iloc[:0]
    other0 = pd.concat([src, fresh.iloc[r.choice(len(fresh), len(test) - kb, replace=False)]], ignore_index=True)
    other0 = other0.drop(columns=["_sid"])
    sh = other0.sample(frac=1.0, random_state=seed)
    planted = sh.index.to_numpy() < kb
    other = transform_corpus(sh.reset_index(drop=True), tf if tf != "rowsubset" else "none", seed)
    return other, planted, kb

if not E["ok"]:
    print("E failed: skipped")
else:
    seeds = [100, 200] if SMOKE else list(range(100, 120)) + list(range(200, 220))
    rhos = (0.0, 0.05) if SMOKE else (0.0, 0.01, 0.05, 0.20)
    seen = {(r["transform"], r["rho"], r["seed"]) for r in map(json.loads, open(TF_FILE))} if os.path.exists(TF_FILE) else set()
    todo = [(tf, rho, s) for s in seeds for tf in (TFS[:3] if SMOKE else TFS) for rho in rhos if (tf, rho, s) not in seen]
    print(len(seen), "done,", len(todo), "to run"); t0 = time.time()
    for i, (tf, rho, s) in enumerate(todo):
        p = plant("lineage", rho, s, transform=tf)
        other, planted, kb = replay(rho, s, tf)
        row = {"transform": tf, "rho": rho, "seed": s, "n": len(other), "planted": int(kb),
               "replay_ok": bool(other.equals(p.other) and kb == p.n_leaked)}
        for m, kw in MODES.items():
            res = L.lineage_matches(other, p.train, seed=0, **kw)
            if res["matched"] is None:
                row[m] = {"aligned": False}
                continue
            mt = np.zeros(len(other), bool); mt[res["index"]] = res["matched"]
            k = int(mt.sum()); lo, hi = wilson(k, len(other))
            row[m] = {"aligned": True, "matched": k, "tp": int((mt & planted).sum()), "C": k / len(other), "lo": lo, "hi": hi}
        with open(TF_FILE, "a") as fh:
            fh.write(json.dumps(row) + "\n")
        if (i + 1) % 100 == 0:
            print(i + 1, f"{time.time() - t0:.0f} s")
    T = pd.DataFrame(list(map(json.loads, open(TF_FILE))))
    print(len(T), "configurations; replay mismatches:", int((~T["replay_ok"]).sum()))


In [ ]:
# 5. B: group-bootstrap 95 % intervals for the sensitivity results (from the saved NB1 reports; no retraining)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
LEVELS = ["row", "dup", "5tuple", "hostpair", "session:60", "session:300", "session:1800", "capture"]
folds, comps = [], []
for f in sorted(glob.glob(f"{NB1}/*_sensitivity.json")):
    ds = os.path.basename(f).split("_")[0]
    d = json.load(open(f))
    fr = pd.DataFrame(d["results"].get("folds", []))
    if len(fr):
        folds.append(fr.assign(dataset=ds))
    tc = pd.DataFrame(d["results"].get("test_composition", []))
    if len(tc):
        comps.append(tc.assign(dataset=ds, model=os.path.basename(f).split("_")[1]))
if not folds:
    print("no NB1 sensitivity reports with fold rows found in", NB1)
    Bsum = None
else:
    F = pd.concat(folds, ignore_index=True)
    F.to_csv(f"{OUT}/B_folds_with_intervals.csv", index=False)
    if comps:
        pd.concat(comps, ignore_index=True).to_csv(f"{OUT}/B_test_composition.csv", index=False)
    rows = []
    for (ds, model, unit), g in F[F["unit"] != "row"].groupby(["dataset", "model", "unit"]):
        ex_pos = bool((g["d_full_lo"] > 0).all()); ex_neg = bool((g["d_full_hi"] < 0).all())
        rows.append({"dataset": ds, "model": model, "unit": unit, "n_folds": len(g), "seeds": int(g["seed"].nunique()),
                     "d_full_mean": round(float(g["d_full"].mean()), 4),
                     "d_full_lo_min": round(float(g["d_full_lo"].min()), 4), "d_full_hi_max": round(float(g["d_full_hi"].max()), 4),
                     "distinguishable": "above 0" if ex_pos else ("below 0" if ex_neg else "no")})
    Bsum = pd.DataFrame(rows)
    Bsum.to_csv(f"{OUT}/B_summary.csv", index=False)
    print(Bsum.to_string(index=False))
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), sharey=False)
    for j, ds in enumerate(["cic", "unsw"]):
        for i, (m, mk) in enumerate([("hgb", "o"), ("rf", "s"), ("lr", "^")]):
            g = F[(F["dataset"] == ds) & (F["model"] == m) & (F["unit"] != "row")]
            for k, (_, r) in enumerate(g.iterrows()):
                x = LEVELS.index(r["unit"]) + (i - 1) * 0.22 + (k % 3 - 1) * 0.05
                ax[j].errorbar(x, r["d_full"], yerr=[[r["d_full"] - r["d_full_lo"]], [r["d_full_hi"] - r["d_full"]]],
                               fmt=mk, ms=3, capsize=1.5, color=f"C{i}", label=m.upper() if k == 0 else None)
        ax[j].axhline(0, color="0.5", lw=0.8); ax[j].set_title({"cic": "CIC-IDS2017 (mirror)", "unsw": "UNSW-NB15 (mirror)"}[ds], fontsize=10)
        ax[j].set_xticks(range(len(LEVELS))); ax[j].set_xticklabels(LEVELS, rotation=40, ha="right", fontsize=8)
    ax[0].set_ylabel(r"$\Delta_{full}$ with 95 % group-bootstrap interval (per seed/fold)")
    ax[1].legend(fontsize=8); fig.tight_layout(); fig.savefig(f"{OUT}/B_fig_sensitivity_ci.pdf"); fig.savefig(f"{OUT}/B_fig_sensitivity_ci.png", dpi=150)


In [ ]:
# 6. X: cross-tool harness, leakdepth runner rerun with 0.1.1 (seeds 100-109; other tools unchanged)
sys.path.insert(0, f"{WD}/bench"); sys.path.insert(0, f"{WD}/bench/compare")
import tempfile
import compare as CMP
import run_leakdepth as RLD
XF = f"{OUT}/X_leakdepth_011.jsonl"
if not E["ok"]:
    print("E failed: skipped")
else:
    seen = {CMP.key(r, "leakdepth") for r in map(json.loads, open(XF))} if os.path.exists(XF) else set()
    cfgs = [c for c in CMP.configs([100] if SMOKE else list(range(100, 110)))]
    todo = [c for c in cfgs if CMP.key(c, "leakdepth") not in seen]
    print(len(seen), "done,", len(todo), "to run")
    for c in todo:
        with tempfile.TemporaryDirectory() as wd:
            job = CMP.materialise(c, wd)
            if job.get("group"):
                pass                                    # leakdepth is not a group tool in the harness: run as defaults
            j = dict(job); j["group"] = None
            t0 = time.time()
            try:
                out = RLD.handle(j); out["error"] = None
            except Exception as e:  # noqa: BLE001
                out = {"detected": None, "error": f"{type(e).__name__}: {e}"[:200]}
            row = {**c, "tool": "leakdepth", "version": L.__version__, "detected": out.get("detected"),
                   "fired": out.get("detail", {}).get("fired"), "error": out.get("error"), "seconds": round(time.time() - t0, 2)}
        with open(XF, "a") as fh:
            fh.write(json.dumps(row, default=str) + "\n")
    X = pd.DataFrame(list(map(json.loads, open(XF))))
    old = {CMP.key(r, "leakdepth"): r["detected"] for r in LD010_ROWS}
    X["old"] = [old.get(CMP.key(r, "leakdepth")) for r in map(json.loads, open(XF))]   # raw records: keeps None (not NaN) in keys
    X["agree"] = X["old"] == X["detected"]
    print(len(X), "configurations; agree with 0.1.0:", int(X["agree"].sum()), "| errors:", int(X["error"].notna().sum()))


In [ ]:
# 7. Compact summary (paste this output back); also nb5/S11_NB5_summary.json
S = {"leakdepth": L.__version__, "E": jload("E.json") if done("E.json") else None}
N = {}
for f in sorted(glob.glob(f"{OUT}/N_*.json")):
    r = json.load(open(f)); g = r["gate"]
    incl = (not g["checked"]) or (g["a_in_b"] <= 0.001 and g["b_in_a"] <= 0.001)
    N[r["pair"]] = {"a": r["a"], "b": r["b"], "rows": r["rows"], "shared_names": len(g["shared"]),
                    "exact_overlap": [round(g["a_in_b"], 5), round(g["b_in_a"], 5)] if g["checked"] else None,
                    "included": incl, "a10[m,n,C,hi]": [c[1:] for c in r["a10"]["containment"]],
                    "frozen[m,n,C,hi]": [c[1:] for c in r["frozen"]["containment"]], "aligned": [r["a10"]["aligned"], r["frozen"]["aligned"]],
                    "evaluable": bool(r["a10"]["containment"]) and bool(r["frozen"]["containment"]),   # no aligned columns -> C undefined
                    "sec": r["seconds_total"]}
S["N"] = N
S["N_criterion_a10_all_included_le_0.01"] = all(c[2] <= 0.01 for v in N.values() if v["included"] and v["evaluable"] for c in v["a10[m,n,C,hi]"]) if N else None
S["N_included_not_evaluable"] = [k for k, v in N.items() if v["included"] and not v["evaluable"]]   # reported, not counted as a pass
S["N_files"] = {k: v for f in sorted(glob.glob(f"{OUT}/N_*.json")) for k, v in json.load(open(f))["files"].items()}
if os.path.exists(TF_FILE):
    T = pd.DataFrame(list(map(json.loads, open(TF_FILE))))
    S["T_replay_mismatch"] = int((~T["replay_ok"]).sum())
    tsum = {}
    for m in MODES:
        recs = [{"transform": r["transform"], "rho": r["rho"], "planted": r["planted"], "s": r["planted"] / r["n"], **r[m]}
                for r in T.to_dict("records") if r[m].get("aligned")]
        V = pd.DataFrame(recs)
        V["cover"] = (V["lo"] <= V["s"]) & (V["s"] <= V["hi"])
        V["recall"] = np.where(V["planted"] > 0, V["tp"] / V["planted"].clip(lower=1), np.nan)
        V["prec"] = np.where(V["matched"] > 0, V["tp"] / V["matched"].clip(lower=1), np.nan)
        V["grp"] = np.where(V["transform"] == "noise", "noise", "non-noise")
        out = {"unaligned_configs": int(len(T) - len(V))}
        for (grp, rho), g in V.groupby(["grp", "rho"]):
            out[f"{grp}|{rho}"] = {"n": len(g), "mean_abs_err_C_minus_s": round(float((g["C"] - g["s"]).abs().mean()), 5),
                                   "ci_coverage": round(float(g["cover"].mean()), 3),
                                   "row_recall_mean": None if rho == 0 else round(float(g["recall"].mean()), 4),
                                   "row_recall_min": None if rho == 0 else round(float(g["recall"].min()), 4),
                                   "precision_min": None if g["prec"].isna().all() else round(float(g["prec"].min()), 4),
                                   "det_rate_by_threshold": [round(float((g["C"] >= t).mean()), 3) for t in THRESH]}
        tsum[m] = out
    S["T[thresholds]"] = THRESH
    S["T"] = tsum
if os.path.exists(f"{OUT}/B_summary.csv"):
    B = pd.read_csv(f"{OUT}/B_summary.csv")
    S["B[dataset|model|unit]=[d_full_mean,lo_min,hi_max,distinguishable]"] = {
        f'{r.dataset}|{r.model}|{r.unit}': [r.d_full_mean, r.d_full_lo_min, r.d_full_hi_max, r.distinguishable] for r in B.itertuples()}
if os.path.exists(XF):
    X = pd.DataFrame(list(map(json.loads, open(XF))))
    old = {CMP.key(r, "leakdepth"): r["detected"] for r in LD010_ROWS}
    X["old"] = [old.get(CMP.key(r, "leakdepth")) for r in map(json.loads, open(XF))]   # raw records keep None keys
    X = X.astype(object).where(X.notna(), None)
    dis = X[X["old"] != X["detected"]]
    S["X"] = {"n": len(X), "agree": int(len(X) - len(dis)), "errors": int(X["error"].notna().sum()),
              "no_010_row": int(X["old"].isna().sum()),
              "disagreements": dis[["leak", "rho", "seed", "transform", "condition", "old", "detected"]].to_dict("records")[:30],
              "detect_rate_011[leak|condition|rho]": {f"{k[0]}|{k[1]}|{k[2]}": round(float(v), 3)
                  for k, v in X.groupby(["leak", "condition", "rho"])["detected"].mean().items()}}
S["runtime_min_this_session"] = round((time.time() - T0) / 60, 1)
jdump(S, "S11_NB5_summary.json")
print(json.dumps(S, separators=(",", ":"), default=str))
